# 03 — Gold: indicatori e verifiche (Silver → Gold)

**Domanda:** dopo il nuovo accordo fiscale (in vigore dal 17.7.2023), il frontalierato italiano ha cambiato traiettoria?

Questo notebook calcola le **tre verifiche** dell'ipotesi:
1. **Traiettoria** — la crescita dei frontalieri italiani è cambiata dopo il 2023?
2. **Gruppo di controllo** — il cambiamento riguarda solo gli italiani, o anche francesi e tedeschi (non toccati dall'accordo)?
3. **Dove e in quali settori** — in quali cantoni e settori si concentra?

**Regola di metodo:** si confrontano sempre trimestri **uguali** (Q2 con Q2), per non farsi ingannare dalla stagionalità.
Il risultato è un'**associazione temporale**, non una prova di causa.

**Input:** tabelle Silver. **Output:** tabelle `gold_*` in `workspace.frontalieri`.


In [ ]:
# --- PARAMETRI ---------------------------------------------------------------
# Tutte le scelte di metodo sono qui, visibili e modificabili.
import pandas as pd
import numpy as np

CATALOG, SCHEMA = "workspace", "frontalieri"

# Rottura: l'accordo entra in vigore il 17.7.2023 -> ultimo trimestre "prima" = 2023-Q2.
# Sensitività: applicazione fiscale dal 1.1.2024 -> ultimo trimestre "prima" = 2023-Q4.
ROTTURE = {"entrata_in_vigore": "2023-Q2", "applicazione_fiscale": "2023-Q4"}
ANNI_PRIMA = 4          # finestra "prima": 4 anni prima della rottura

# Gruppi da confrontare (paese di residenza + cantoni di lavoro)
GRUPPI = {
    "IT -> TI+GR+VS (trattato)":   ("IT", ["21", "18", "23"]),   # dove valgono le regole dell'accordo
    "IT -> Svizzera":              ("IT", ["_T"]),
    "FR -> GE+VD (controllo)":     ("FR", ["25", "22"]),         # non toccati dall'accordo IT-CH
    "FR -> Svizzera":              ("FR", ["_T"]),
    "DE -> Svizzera (controllo)":  ("DE", ["_T"]),
    "Tutti -> Svizzera":           ("_T", ["_T"]),
}
TRATTATO, CONTROLLO = "IT -> TI+GR+VS (trattato)", "FR -> GE+VD (controllo)"


In [ ]:
# --- 1. LETTURA DELLE TABELLE SILVER -----------------------------------------
cant = spark.table(f"{CATALOG}.{SCHEMA}.silver_frontalieri_cantone").toPandas()
sett = spark.table(f"{CATALOG}.{SCHEMA}.silver_frontalieri_settori").toPandas()
print("silver cantone:", cant.shape, "| silver settori:", sett.shape)
print("ultimo trimestre disponibile:", cant["trimestre"].max())


In [ ]:
# --- 2. FUNZIONI DI CALCOLO (riutilizzabili in progetti futuri) --------------
def q_diff(a, b):
    """Numero di trimestri tra due trimestri 'AAAA-Qn' (b - a)."""
    ya, qa = int(a[:4]), int(a[-1]); yb, qb = int(b[:4]), int(b[-1])
    return (yb - ya) * 4 + (qb - qa)

def q_shift(q, n):
    """Sposta un trimestre di n trimestri (n negativo = indietro)."""
    y, k = int(q[:4]), int(q[-1]) - 1 + n
    return f"{y + k // 4}-Q{k % 4 + 1}"

def serie_gruppo(df, paese, cantoni):
    """Somma i frontalieri di un paese su un insieme di cantoni, per trimestre."""
    s = df[(df.paese_cod == paese) & (df.cantone_cod.isin(cantoni))]
    return s.groupby("trimestre")["frontalieri"].sum().sort_index()

def crescita_annua(serie, da, a):
    """Crescita media annua composta (%) tra due trimestri."""
    anni = q_diff(da, a) / 4
    return 100 * ((serie[a] / serie[da]) ** (1 / anni) - 1)


In [ ]:
# --- 3. GOLD A: serie trimestrali per gruppo, con indici e variazioni --------
ultimo = cant["trimestre"].max()
righe = []
for nome, (paese, cantoni) in GRUPPI.items():
    s = serie_gruppo(cant, paese, cantoni)
    df = pd.DataFrame({"gruppo": nome, "trimestre": s.index, "frontalieri": s.values})
    df["var_annua_pct"] = 100 * (df["frontalieri"] / df["frontalieri"].shift(4) - 1)   # stesso trimestre anno prima
    df["indice_2023Q2"] = 100 * df["frontalieri"] / s["2023-Q2"]                      # 2023-Q2 = 100
    df["indice_2019"] = 100 * df["frontalieri"] / s[[f"2019-Q{i}" for i in range(1, 5)]].mean()  # media 2019 = 100
    df["periodo"] = np.where(df["trimestre"] <= ROTTURE["entrata_in_vigore"], "prima", "dopo")
    righe.append(df)
gold_serie = pd.concat(righe, ignore_index=True)
gold_serie["anno"] = gold_serie["trimestre"].str[:4].astype(int)
print(gold_serie.shape)
display(gold_serie[gold_serie.trimestre == ultimo].round(1))


### Verifiche 1 e 2 — traiettoria prima/dopo e confronto con i gruppi di controllo

Per ogni gruppo: **crescita media annua** nei 4 anni prima della rottura e nel periodo dopo, fino all'ultimo dato.
Poi la **differenza nelle differenze**: quanto il trattato (italiani nei cantoni dell'accordo) ha rallentato *in più* rispetto al controllo (francesi a GE+VD).
Se è vicina a zero, il rallentamento non è specifico degli italiani.


In [ ]:
# --- 4. GOLD B: verifica traiettoria + differenza nelle differenze ------------
# I punti di partenza/arrivo sono trimestri omologhi (es. Q2 -> Q2) o, per il "dopo",
# l'ultimo dato disponibile: la crescita è annualizzata quindi resta confrontabile.
righe = []
for nome_rottura, rottura in ROTTURE.items():
    inizio = q_shift(rottura, -4 * ANNI_PRIMA)
    for nome, (paese, cantoni) in GRUPPI.items():
        s = serie_gruppo(cant, paese, cantoni)
        prima = crescita_annua(s, inizio, rottura)
        dopo = crescita_annua(s, rottura, ultimo)
        righe.append({"rottura": nome_rottura, "trimestre_rottura": rottura, "gruppo": nome,
                      "finestra_prima": f"{inizio} -> {rottura}", "finestra_dopo": f"{rottura} -> {ultimo}",
                      "crescita_annua_prima_pct": prima, "crescita_annua_dopo_pct": dopo,
                      "cambio_pp": dopo - prima})
gold_verifica = pd.DataFrame(righe)

# Differenza nelle differenze (in punti percentuali di crescita annua)
did = (gold_verifica.pivot(index="rottura", columns="gruppo", values="cambio_pp")
       .assign(diff_nelle_diff_pp=lambda d: d[TRATTATO] - d[CONTROLLO])[[TRATTATO, CONTROLLO, "diff_nelle_diff_pp"]])

display(gold_verifica.round(2))
print("Differenza nelle differenze (pp): negativa = gli italiani hanno rallentato PIÙ del controllo")
display(did.round(2))


### Verifica 3 — dove e in quali settori

Variazione dei frontalieri **residenti in Italia** tra il trimestre prima dell'accordo (2023-Q2) e l'ultimo dato, per cantone e per sezione economica (Ticino).
Accanto, la stessa misura per i **francesi** come riferimento.


In [ ]:
# --- 5. GOLD C: cantoni e settori --------------------------------------------
R = ROTTURE["entrata_in_vigore"]

# 5a. Per cantone (italiani), con la crescita prima come riferimento
righe = []
for cod, nome in [("21", "Ticino"), ("18", "Grigioni"), ("23", "Vallese"), ("_T", "Svizzera")]:
    s = serie_gruppo(cant, "IT", [cod])
    righe.append({"cantone": nome, "frontalieri_2023Q2": s[R], "frontalieri_ultimo": s[ultimo], "ultimo_trimestre": ultimo,
                  "variazione_assoluta": s[ultimo] - s[R],
                  "variazione_pct": 100 * (s[ultimo] / s[R] - 1),
                  "crescita_annua_prima_pct": crescita_annua(s, q_shift(R, -16), R)})
gold_cantoni = pd.DataFrame(righe)
display(gold_cantoni.round(1))

# 5b. Per sezione economica in Ticino: italiani vs totale, prima e dopo
SEZ = list("ABCDEFGHIJKLMNOPQRSTU")
ti = sett[(sett.cantone_cod == "21") & (sett.noga_cod.isin(SEZ + ["1", "2", "3", "_T"]))]
piv = ti.pivot_table(index=["noga_cod", "noga_desc"], columns=["paese_cod", "trimestre"],
                     values="frontalieri", aggfunc="sum")
inizio = q_shift(R, -16)
g = pd.DataFrame({
    "it_2023Q2":         piv[("IT", R)],
    "it_ultimo":         piv[("IT", ultimo)],
    "it_var_dopo_pct":   100 * (piv[("IT", ultimo)] / piv[("IT", R)] - 1),
    "it_var_prima_pct":  100 * (piv[("IT", R)] / piv[("IT", inizio)] - 1),
    "it_var_assoluta":   piv[("IT", ultimo)] - piv[("IT", R)],
}).reset_index()
g["ultimo_trimestre"] = ultimo
g["tipo"] = np.where(g.noga_cod.isin(["1", "2", "3", "_T"]), "settore/totale", "sezione")
gold_settori_ticino = g.sort_values(["tipo", "it_var_assoluta"])
display(gold_settori_ticino.round(1))


In [ ]:
# --- 6. CONTROLLI DI QUALITÀ ---------------------------------------------------
# Bloccanti: se falliscono, nessuna tabella Gold viene scritta.
chk = gold_serie[(gold_serie.gruppo == "IT -> Svizzera") & (gold_serie.trimestre == ultimo)]["frontalieri"].iloc[0]
src = cant[(cant.paese_cod == "IT") & (cant.cantone_cod == "_T") & (cant.trimestre == ultimo)]["frontalieri"].iloc[0]
print(f"[bloccante] Gold = Silver (IT, Svizzera, {ultimo}): {chk:.2f} vs {src:.2f}")
assert abs(chk - src) < 1e-6

base = gold_serie[gold_serie.trimestre == "2023-Q2"]["indice_2023Q2"]
print("[bloccante] indice 2023-Q2 = 100 per tutti i gruppi:", bool(np.allclose(base, 100)))
assert np.allclose(base, 100)

nan = gold_verifica[["crescita_annua_prima_pct", "crescita_annua_dopo_pct"]].isna().sum().sum()
print("[bloccante] valori mancanti nella verifica:", nan)
assert nan == 0

somma_sez = gold_settori_ticino[gold_settori_ticino.tipo == "sezione"]["it_ultimo"].sum()
tot = gold_settori_ticino[gold_settori_ticino.noga_cod == "_T"]["it_ultimo"].iloc[0]
print(f"[info] somma sezioni vs totale Ticino (IT): {somma_sez:.0f} vs {tot:.0f} ({100*(somma_sez/tot-1):+.2f}%)")


In [ ]:
# --- 7. GRAFICO: indice 2023-Q2 = 100, trattato vs controlli ----------------
# Un solo asse (indice comune), linea verticale all'entrata in vigore, etichette dirette.
import matplotlib.pyplot as plt

SERIE = [(TRATTATO, "#2a78d6"), (CONTROLLO, "#eb6834"), ("DE -> Svizzera (controllo)", "#1baf7a")]
d = gold_serie[gold_serie.trimestre >= "2019-Q1"]
trim = sorted(d.trimestre.unique())
x = {q: i for i, q in enumerate(trim)}

fig, ax = plt.subplots(figsize=(10, 5.5))
for nome, col in SERIE:
    s = d[d.gruppo == nome].set_index("trimestre")["indice_2023Q2"]
    ax.plot([x[q] for q in s.index], s.values, color=col, lw=2, label=nome)
    ax.annotate(f"{s.iloc[-1]:.1f}", (x[s.index[-1]], s.iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color="#333333")
ax.axvline(x["2023-Q3"], color="#888888", ls="--", lw=1)
ax.text(x["2023-Q3"] + 0.3, ax.get_ylim()[1], "Accordo in vigore\n(17.7.2023)", va="top", fontsize=9, color="#555555")
ax.axhline(100, color="#cccccc", lw=0.8)
ax.set_xticks([x[q] for q in trim if q.endswith("Q1")])
ax.set_xticklabels([q[:4] for q in trim if q.endswith("Q1")])
ax.set_ylabel("Indice (2023-Q2 = 100)")
ax.set_title("Frontalieri: italiani nei cantoni dell'accordo vs gruppi di controllo")
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#eeeeee")
ax.legend(frameon=False, loc="upper left", fontsize=9)
fig.text(0.01, 0.01, f"Fonte: UST, statistica dei frontalieri (DF_GGS_1). Dati trimestrali fino a {ultimo}.",
         fontsize=8, color="#666666")
plt.tight_layout()
plt.show()


In [ ]:
# --- 8. SCRITTURA TABELLE GOLD -----------------------------------------------
for df, nome in [(gold_serie, "gold_serie_gruppi"), (gold_verifica, "gold_verifica_traiettoria"),
                 (gold_cantoni, "gold_cantoni_italiani"), (gold_settori_ticino, "gold_settori_ticino")]:
    out = df.copy()
    out.columns = [str(c).replace("-", "") for c in out.columns]     # nomi colonna validi in SQL
    (spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{nome}"))
    print(f"OK {CATALOG}.{SCHEMA}.{nome}: {len(out)} righe")
